In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
df_train = pd.read_parquet("../data/cleaned/train_data.parquet")

In [ ]:
print("Starting Feature Engineering: Kabure + Artgor Synthesis...")

# We use np.log1p (which is log(1 + x)) to safely handle any zero values
df_train['TransactionAmt_Log'] = np.log1p(df_train['TransactionAmt'])

print("Successfully applied Log Transformation to TransactionAmt.")

In [ ]:
print("Applying Cyclical Time Encoding...")

# 1. Re-extract the raw hours and days 
raw_hour = np.floor((df_train['TransactionDT'] / 3600) % 24)
raw_day = np.floor((df_train['TransactionDT'] / (3600 * 24) - 1) % 7)

# 2. Cyclical Encoding for Hour (24-hour period)
df_train['Hour_Sin'] = np.sin(2 * np.pi * raw_hour / 24.0)
df_train['Hour_Cos'] = np.cos(2 * np.pi * raw_hour / 24.0)

# 3. Cyclical Encoding for Weekday (7-day period)
df_train['Day_Sin'] = np.sin(2 * np.pi * raw_day / 7.0)
df_train['Day_Cos'] = np.cos(2 * np.pi * raw_day / 7.0)

print("Time features successfully converted to cyclical coordinates.")

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: The Raw Data (Highly Skewed - Bad for PyTorch)
sns.histplot(df_train['TransactionAmt'], bins=50, ax=ax[0], color='red', kde=True)
ax[0].set_title('Raw TransactionAmt (XGBoost can handle this)', fontweight='bold')
ax[0].set_xlim(0, 1000) # Limiting X so we can actually see the spike

# Plot 2: The Log Transformed Data (Smooth - Perfect for PyTorch)
sns.histplot(df_train['TransactionAmt_Log'], bins=50, ax=ax[1], color='green', kde=True)
ax[1].set_title('Log Transformed TransactionAmt_Log (PyTorch Ready)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
print("Starting GNN-Specific Preparation...")

# ---------------------------------------------------------
# 1. Standard Scaling Numerical Features
# ---------------------------------------------------------
# We isolate all the numeric columns (excluding our encoded categorical ones)
# Idea is - that we scale all the features like C,D,V except id's
# 1. Define columns that are numbers, but are actually Categories/IDs
pseudo_numeric_cats = [
    'TransactionID', 'isFraud', 'TransactionDT', 
    'card1', 'card2', 'card3', 'card5', 
    'addr1', 'addr2'
]

# 2. Grab all numeric columns
all_numerics = [col for col in df_train.columns if df_train[col].dtype in ['float32', 'float64', 'int16', 'int32', 'int64']]

# 3. Strictly filter out the pseudo-numeric categories
# C1 - C14, D1-D15, V1-V339, TransactionAmt
numeric_cols_to_scale = [col for col in all_numerics if col not in pseudo_numeric_cats]

# 4. Scale ONLY the continuous mathematical features (like TransactionAmt, C-features, D-features)
print(f"Scaling {len(numeric_cols_to_scale)} continuous numeric features...")
scaler = StandardScaler()
df_train[numeric_cols_to_scale] = scaler.fit_transform(df_train[numeric_cols_to_scale])

print("Standardization complete. IDs remain intact. No columns left now.")

In [ ]:
# ---------------------------------------------------------
# 2. GRAPH EDA: Visualizing Node Connections (Shared Cards)
# ---------------------------------------------------------
# How many times is the exact same credit card used?
card_frequencies = df_train['card1'].value_counts()

plt.figure(figsize=(10, 6))
# We limit the x-axis to 50, otherwise the few cards used 10,000 times ruin the plot
sns.histplot(card_frequencies[card_frequencies < 50], bins=50, color='purple', kde=False)

plt.title('Node Degree Distribution (Shared "card1" edges)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Transactions per Card (Node Degree)', fontsize=12)
plt.ylabel('Count of Unique Cards', fontsize=12)
plt.show()

In [ ]:
print("Plotting PCA Cumulative Variance...")
v_cols = [col for col in df_train.columns if col.startswith('V')]

# 1. Fit PCA with a wider net (150 components) just to visualize the curve
# We don't transform the dataframe yet, we just fit it to get the math
pca_full = PCA(n_components=150, random_state=42)
pca_full.fit(df_train[v_cols])

# 2. Calculate the cumulative sum of the variance
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

# 3. Plot the curve
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

plt.plot(range(1, 151), cumulative_variance, marker='o', linestyle='--', color='b', markersize=3)

# 4. Add threshold lines to explicitly justify your choice
plt.axhline(y=0.90, color='red', linestyle='-', linewidth=2, label='90% Variance Threshold')
plt.axvline(x=75, color='green', linestyle='-', linewidth=2, label='75 Components (92.9% Retained)')

# 5. Labels and Titles
plt.title('PCA Cumulative Explained Variance (V-Features)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Principal Components', fontsize=12)
plt.ylabel('Cumulative Variance Retained', fontsize=12)
plt.legend(loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
print("Starting PCA Compression on V-Features...")

# 1. Isolate the V-Features
# V-features are named V1 through V339
print(f"Found {len(v_cols)} V-features.")

# 2. Initialize PCA
# We want to compress them down to exactly 30 components
n_components = 30
pca = PCA(n_components=n_components, random_state=42)

# 3. Fit and Transform the V-features
# Note: PCA requires scaled data, and we already ran StandardScaler earlier
v_pca = pca.fit_transform(df_train[v_cols])

# 4. Create new column names for our compressed features
pca_cols = [f'V_PCA_{i}' for i in range(1, n_components + 1)]

# 5. Add the new compressed columns to our main dataframe
v_pca_df = pd.DataFrame(v_pca, columns=pca_cols, index=df_train.index)
df_train = pd.concat([df_train, v_pca_df], axis=1)

# 6. Drop the original 339 V-columns to save memory
df_train = df_train.drop(columns=v_cols)

print(f"Successfully compressed {len(v_cols)} columns into {n_components} components.")
print(f"Total variance retained by these {n_components} components: {sum(pca.explained_variance_ratio_) * 100:.2f}%")

In [ ]:
# Convert sin/cos back to radians, then back to the 24-hour scale
decoded_hours = np.round((np.arctan2(df_train['Hour_Sin'], df_train['Hour_Cos']) * 24 / (2 * np.pi)))

# arctan2 can return negative hours (e.g., -2 instead of 22), so we modulo 24 to fix it
decoded_hours = decoded_hours % 24

print(decoded_hours.head())

In [ ]:
print("Saving engineered dataset to Parquet format...")

# Save as parquet to preserve int8/float32 data types and save disk space
df_train.to_parquet("../data/processed/train_engineered.parquet", index=False)